# Profiling Fase 2 — Soporte a Stage, Anonimización y Modelo Dimensional
### Proyecto: Plataforma Analítica de Mortalidad (PNUD/MSPAS) · Databricks (serverless)

**Continuación del profiling base.** Este notebook agrega los chequeos que la **Fase 2**
necesita y que el primer profiling no cubrió, todos **read-only**:

1. Semántica real de `Perdif` vs `Puedif` (corrige el mapeo del plan de anonimización).
2. Edad × Perdif: cuantifica edades en meses/días e ignoradas (corrige la generalización de edad).
3. **Factibilidad de k-anonimato** (k=5 y k=10): % de registros que se suprimirían — define el
   trade-off privacidad/utilidad del plan de anonimización (EU Data Act / GDPR).
4. Distribución CIE-10 por capítulo y categoría (soporta `dim_causa` y la generalización CIE-10).
5. Cobertura de `Areag` (urbano/rural) por periodo — limitación para el análisis pre/post-COVID.

Sigue leyendo Delta directo (`spark.table("bronze.…")`), sin S3 ni credenciales.


In [0]:
# ── 0. SETUP ─────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
pd.set_option("display.width", 160)

SCHEMA = "bronze"
INE = ["xlsx_ine", "sav_ine_legacy"]

def norm(colname):
    # quita artefacto float ".0" y normaliza nan/vacío a NULL (clave para agrupar bien)
    c = F.col(f"`{colname}`")
    c = F.regexp_replace(c, r"\.0$", "")
    return F.when(F.lower(F.trim(c)).isin("", "nan", "none", "null"), None).otherwise(c)

print("Setup OK. Tablas INE:", INE)


Setup OK. Tablas INE: ['xlsx_ine', 'sav_ine_legacy']


In [0]:
# ── 1. SEMÁNTICA: Perdif (¿unidad de edad?) vs Puedif (¿pueblo/etnia?) ────────
# Evidencia para corregir el plan de anonimización (que usa 'perdif' como etnia).
for t in INE:
    df = spark.table(f"{SCHEMA}.{t}")
    print("\n" + "="*70)
    print(f"{t} — Perdif  (cardinalidad esperada baja = unidad de edad)")
    print("="*70)
    df.groupBy(F.expr("regexp_replace(Perdif,'\\\\.0$','')").alias("Perdif")) \
      .count().orderBy(F.desc("count")).show(truncate=False)
    print(f"{t} — Puedif  (cardinalidad ~6 = pueblo/etnia)")
    df.groupBy(F.expr("regexp_replace(Puedif,'\\\\.0$','')").alias("Puedif")) \
      .count().orderBy(F.desc("count")).show(truncate=False)



xlsx_ine — Perdif  (cardinalidad esperada baja = unidad de edad)
+------+------+
|Perdif|count |
+------+------+
|3     |626139|
|1     |24168 |
|2     |21760 |
|9     |1997  |
+------+------+

xlsx_ine — Puedif  (cardinalidad ~6 = pueblo/etnia)
+------+------+
|Puedif|count |
+------+------+
|4     |363398|
|1     |189911|
|9     |110328|
|5     |9391  |
|3     |826   |
|2     |210   |
+------+------+


sav_ine_legacy — Perdif  (cardinalidad esperada baja = unidad de edad)
+------+------+
|Perdif|count |
+------+------+
|3     |219673|
|1     |12254 |
|2     |11940 |
|9     |1300  |
+------+------+

sav_ine_legacy — Puedif  (cardinalidad ~6 = pueblo/etnia)
+------+------+
|Puedif|count |
+------+------+
|4     |126930|
|1     |67866 |
|9     |47678 |
|5     |2468  |
|2     |141   |
|3     |84    |
+------+------+



In [0]:
# ── 2. EDAD × PERDIF: ¿la edad está en años, meses o días? ────────────────────
# Si para Perdif=1/2 la edad no son años, la generalización por rangos DEBE usar Perdif.
for t in INE:
    df = spark.table(f"{SCHEMA}.{t}")
    print("\n" + "="*70)
    print(f"{t} — rango de Edadif por unidad (Perdif)")
    print("="*70)
    (df.withColumn("Perdif", F.expr("regexp_replace(Perdif,'\\\\.0$','')"))
       .withColumn("ed", F.col("Edadif").cast("double"))
       .groupBy("Perdif")
       .agg(F.min("ed").alias("edad_min"),
            F.max("ed").alias("edad_max"),
            F.count("*").alias("registros"))
       .orderBy("Perdif").show(truncate=False))
print("Interpretación esperada: Perdif=3 -> años (0..~120); 1/2 -> meses/días;")
print("9 -> edad ignorada (valor centinela). Confirmar etiquetas con el diccionario.")



xlsx_ine — rango de Edadif por unidad (Perdif)
+------+--------+--------+---------+
|Perdif|edad_min|edad_max|registros|
+------+--------+--------+---------+
|1     |0.0     |29.0    |24168    |
|2     |1.0     |11.0    |21760    |
|3     |1.0     |120.0   |626139   |
|9     |999.0   |999.0   |1997     |
+------+--------+--------+---------+


sav_ine_legacy — rango de Edadif por unidad (Perdif)
+------+--------+--------+---------+
|Perdif|edad_min|edad_max|registros|
+------+--------+--------+---------+
|1     |0.0     |29.0    |12254    |
|2     |1.0     |11.0    |11940    |
|3     |1.0     |116.0   |219673   |
|9     |999.0   |999.0   |1300     |
+------+--------+--------+---------+

Interpretación esperada: Perdif=3 -> años (0..~120); 1/2 -> meses/días;
9 -> edad ignorada (valor centinela). Confirmar etiquetas con el diccionario.


In [0]:
from pyspark.sql import functions as F
import pandas as pd

# ── 3. FACTIBILIDAD DE k-ANONIMATO ─────────────────────────────────────────
# Construye un mini-conformado y mide cuántos registros caen en grupos con menos de k.
# Compatible con Databricks Serverless: NO usa cache(), persist(), checkpoint() ni CACHE TABLE.

def conformar(t):
    df = spark.table(f"{SCHEMA}.{t}")

    df = (
        df
        .withColumn(
            "Mupocu",
            F.lpad(
                F.regexp_replace(F.col("Mupocu").cast("string"), r"\.0$", ""),
                4,
                "0"
            )
        )
        .withColumn("Anioocu", norm("Añoocu"))
        .withColumn("Puedif", norm("Puedif"))
        .withColumn("Caudef", F.upper(F.trim(F.col("Caudef").cast("string"))))
    )

    return df.select("Mupocu", "Anioocu", "Puedif", "Caudef")


defun = conformar("xlsx_ine").unionByName(conformar("sav_ine_legacy"))

total = defun.agg(F.count(F.lit(1)).alias("total")).first()["total"]

print(f"Total registros conformados: {total:,}\n")


def pct_suprimido(cols, k):
    """
    Para un conjunto de cuasi-identificadores, agrupa los registros,
    cuenta cuántos registros hay por grupo y suma los registros de grupos con n < k.
    """
    row = (
        defun
        .groupBy(*cols)
        .agg(F.count(F.lit(1)).alias("n"))
        .filter(F.col("n") < F.lit(k))
        .agg(F.coalesce(F.sum("n"), F.lit(0)).alias("suprimidos"))
        .first()
    )

    sup = int(row["suprimidos"]) if row else 0
    pct = round(100 * sup / total, 2) if total else 0

    return sup, pct


filas = []

combos = {
    "municipio+año": ["Mupocu", "Anioocu"],
    "municipio+año+etnia(Puedif)": ["Mupocu", "Anioocu", "Puedif"],
    "municipio+año+causa": ["Mupocu", "Anioocu", "Caudef"],
}

for nombre, cols in combos.items():
    for k in [5, 10]:
        s, p = pct_suprimido(cols, k)
        filas.append({
            "cuasi-identificador": nombre,
            "k": k,
            "suprimidos": s,
            "pct": p
        })

resultado = pd.DataFrame(filas)

print(resultado.to_string(index=False))
print("\nLectura: el % suprimido es el costo de utilidad del k-anonimato.")
print("Sirve para justificar el umbral: k=5 como mínimo; k=10 como opción más conservadora.")

Total registros conformados: 919,231

        cuasi-identificador  k  suprimidos   pct
              municipio+año  5          10  0.00
              municipio+año 10          56  0.01
municipio+año+etnia(Puedif)  5        5923  0.64
municipio+año+etnia(Puedif) 10       15571  1.69
        municipio+año+causa  5      271046 29.49
        municipio+año+causa 10      391258 42.56

Lectura: el % suprimido es el costo de utilidad del k-anonimato.
Sirve para justificar el umbral: k=5 como mínimo; k=10 como opción más conservadora.


In [0]:
# ── 4. CIE-10: distribución por capítulo (1 char) y categoría (3 chars) ──────
# Soporta dim_causa del DW y la generalización CIE-10 del plan (Silver=3, Gold=1).
caudef = (spark.table(f"{SCHEMA}.xlsx_ine").select("Caudef")
          .unionByName(spark.table(f"{SCHEMA}.sav_ine_legacy").select("Caudef"))
          .withColumn("c", F.upper(F.trim(F.col("Caudef")))))
caudef = (caudef.withColumn("capitulo", F.substring("c", 1, 1))
                .withColumn("categoria", F.substring("c", 1, 3)))
print("Distribución por capítulo CIE-10 (1 char):")
caudef.groupBy("capitulo").count().orderBy(F.desc("count")).show(30, truncate=False)
print("Códigos completos distintos :", caudef.select("c").distinct().count())
print("Categorías distintas (3 chars):", caudef.select("categoria").distinct().count())
print("Capítulos distintos (1 char)  :", caudef.select("capitulo").distinct().count())


Distribución por capítulo CIE-10 (1 char):
+--------+------+
|capitulo|count |
+--------+------+
|I       |154438|
|R       |124564|
|E       |96950 |
|J       |84848 |
|C       |80675 |
|X       |75724 |
|K       |70863 |
|N       |39365 |
|A       |38549 |
|P       |28114 |
|U       |24846 |
|W       |18025 |
|G       |17690 |
|V       |17083 |
|Q       |12956 |
|D       |12001 |
|F       |6724  |
|B       |4225  |
|M       |3563  |
|Y       |3295  |
|O       |2611  |
|L       |2019  |
|H       |103   |
+--------+------+

Códigos completos distintos : 3087
Categorías distintas (3 chars): 1102
Capítulos distintos (1 char)  : 23


In [0]:
# ── 5. COBERTURA DE Areag (urbano/rural) por periodo ─────────────────────────
# Areag solo existe en 2015-2017 -> limitación para el análisis urbano/rural pre/post-COVID.
for t in INE:
    df = spark.table(f"{SCHEMA}.{t}")
    print("\n" + "="*60)
    if "Areag" in df.columns:
        print(f"{t}: Areag PRESENTE")
        df.groupBy(F.expr("regexp_replace(Areag,'\\\\.0$','')").alias("Areag")).count() \
          .orderBy(F.desc("count")).show(truncate=False)
    else:
        print(f"{t}: SIN columna Areag (urbano/rural NO disponible en este periodo)")
print("Implicación: el desglose urbano/rural solo es posible 2015-2017 (pre-COVID).")
print("Para 2018-2024 Areag = NULL; documentarlo como limitación del análisis.")



xlsx_ine: SIN columna Areag (urbano/rural NO disponible en este periodo)

sav_ine_legacy: Areag PRESENTE
+-----+------+
|Areag|count |
+-----+------+
|1    |137056|
|2    |104322|
|9    |3789  |
+-----+------+

Implicación: el desglose urbano/rural solo es posible 2015-2017 (pre-COVID).
Para 2018-2024 Areag = NULL; documentarlo como limitación del análisis.


In [0]:
# ── 6. CONFIRMACIÓN DE ETIQUETAS DEL DICCIONARIO (bronze.gdrive_docs) ─────────
# Reestructura el diccionario (forward-fill del nombre de variable) y muestra los
# bloques código->etiqueta de las variables clave para la anonimización / DW.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

d = (spark.table("bronze.gdrive_docs")
        .toDF("variable", "codigo", "etiqueta", "_arch", "_fte")
        .filter(~((F.col("codigo") == "Código") & (F.col("etiqueta") == "Etiqueta"))))

# forward-fill del nombre de variable (celdas combinadas de Excel)
w = Window.orderBy(F.monotonically_increasing_id()).rowsBetween(Window.unboundedPreceding, 0)
d = d.withColumn("variable", F.last("variable", ignorenulls=True).over(w)) \
     .filter(F.col("codigo").isNotNull())

# 1) Catálogo de variables disponibles (para mapear nombre <-> columna INE)
print("=== Variables presentes en el diccionario ===")
d.select("variable").distinct().orderBy("variable").show(100, truncate=False)

# 2) Bloques código->etiqueta de las variables clave (búsqueda por palabra clave)
claves = ["sexo", "pueblo", "area", "área", "edad", "period", "perío",
          "asist", "ocurr", "certif", "civil", "escolar"]
patron = "(?i)(" + "|".join(claves) + ")"
print("\n=== Códigos y etiquetas de variables clave ===")
(d.filter(F.col("variable").rlike(patron))
   .select("variable", "codigo", "etiqueta")
   .orderBy("variable", "codigo")
   .show(200, truncate=False))

=== Variables presentes en el diccionario ===


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------------------------------------+
|variable                                    |
+--------------------------------------------+
|Asistencia recibida                         |
|Año de registro                             |
|Año ocurrencia                              |
|Causa de defuncion                          |
|Departamento de nacimiento del difunto(a)   |
|Departamento de ocurrencia                  |
|Departamento de registro                    |
|Departamento de residencia del difunto(a)   |
|Día de ocurrencia                           |
|Edad del difunto(a)                         |
|Escolaridad del difunto(a)                  |
|Estado civil del difunto(a)                 |
|Mes de ocurrencia                           |
|Mes de registro                             |
|Municipio de nacimiento del difunto(a)      |
|Municipio de ocurrencia                     |
|Municipio de registro                       |
|Municipio de residencia del difunto(a)      |
|Nacionalidad